# Per-Subject DSAR Erasure · 4. Orchestrate & validate (the monthly job)

End-to-end driver that ties the pieces together the way the **monthly** production job would (~10 requests/run):

1. **Erase** — run the targeted in-place UPDATEs for all PENDING requests across all registered tables
   (same logic as `02`).
2. **Purge** — REORG + VACUUM the erased tables and scrub the request table (same as `03`).
3. **Validate** — scan every registered table and assert **no trace** of any processed subject remains.
4. **Report** — per-request status + a final PASS/FAIL.

Deploy this notebook as a **monthly Databricks Job** (schedule: monthly). It reads config from the registry
and requests from `dsar_request`, so onboarding a new table = one registry row, no code change.

> This notebook contains the full logic inline so it can run standalone as the job entry point. The
> standalone notebooks `02`/`03` are the same steps broken out for demo/inspection.

## 0. Configuration

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema",  "allegiant_air_dsar", "2 Schema")
dbutils.widgets.text("redaction_token", "***REDACTED***", "3 Redaction token")
dbutils.widgets.dropdown("do_purge", "true", ["true","false"], "4 Physically VACUUM after erase")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
TOKEN   = dbutils.widgets.get("redaction_token")
DO_PURGE = dbutils.widgets.get("do_purge") == "true"
FQ      = f"{CATALOG}.{SCHEMA}"
from collections import defaultdict
print("Schema:", FQ)

## 1. Load config (requests + registry) and helpers

In [0]:
requests = [r.asDict() for r in spark.sql(f"SELECT * FROM {FQ}.dsar_request WHERE status='PENDING'").collect()]
reg = [r.asDict() for r in spark.sql(f"SELECT * FROM {FQ}.pii_column_registry").collect()]
by_table = defaultdict(list)
for r in reg: by_table[r["table_name"]].append(r)

def sqlstr(v): return "'" + (v or "").replace("'", "''") + "'"

def json_mask_expr(col):
    e = f"regexp_replace({col}, '(\"name\" *: *)\"[^\"]*\"', '$1\"***REDACTED***\"')"
    e = f"regexp_replace({e}, '([?&](firstName|lastName|email)=)[^&#\"]*', '$1***REDACTED***')"
    return e

def match_predicate(table, req):
    ids = [r for r in by_table[table] if r["is_identifier"]]
    if not ids: return None
    preds = []
    for r in ids:
        role, col = r["identifier_role"], r["column_name"]
        if r["erase_strategy"] == "json_key":
            preds += [f"{col} LIKE '%' || {sqlstr(req['email'])} || '%'",
                      f"{col} LIKE '%' || {sqlstr(req['first_name'])} || '%'",
                      f"{col} LIKE '%' || {sqlstr(req['last_name'])} || '%'"]
        elif role == "email":      preds.append(f"{col} = {sqlstr(req['email'])}")
        elif role == "first_name": preds.append(f"{col} = {sqlstr(req['first_name'])}")
        elif role == "last_name":  preds.append(f"{col} = {sqlstr(req['last_name'])}")
        elif role == "full_name":  preds.append(f"{col} = {sqlstr(req['first_name']+' '+req['last_name'])}")
    return " AND ".join(preds)

def set_clause(table):
    sets = []
    for r in by_table[table]:
        col, strat = r["column_name"], r["erase_strategy"]
        if strat in ("scalar_token","pnr_token"): sets.append(f"{col} = {sqlstr(TOKEN)}")
        elif strat == "json_key": sets.append(f"{col} = {json_mask_expr(col)}")
    return ", ".join(sets)

print(len(requests), "pending;", len(by_table), "tables")

## 2. STEP 1 — Erase (targeted, in place)

In [0]:
erased_subjects = [dict(r) for r in requests]  # snapshot before we scrub the request table
audit = []
for req in requests:
    for table in by_table:
        pred = match_predicate(table, req)
        if not pred: continue
        fqt = f"{FQ}.{table}"
        n = spark.sql(f"SELECT count(*) c FROM {fqt} WHERE {pred}").collect()[0]["c"]
        if n: spark.sql(f"UPDATE {fqt} SET {set_clause(table)} WHERE {pred}")
        audit.append((req["request_id"], table, n))
print("erase done; rows changed:", sum(a[2] for a in audit))

## 3. STEP 2 — Purge (physical) + scrub request table
Uses the serverless-safe zero-retention approach: table property `delta.deletedFileRetentionDuration =
interval 0 hours` + `REORG … APPLY (PURGE)` + plain `VACUUM` (no session conf). The request table is scrubbed
of raw identifiers *before* it too is purged.

In [0]:
def purge_table(fqt, do_vacuum=True):
    spark.sql(f"ALTER TABLE {fqt} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")
    try: spark.sql(f"REORG TABLE {fqt} APPLY (PURGE)")
    except Exception as e: print("REORG skip", fqt, str(e).splitlines()[0])
    if do_vacuum: spark.sql(f"VACUUM {fqt}")   # no RETAIN clause -> honours the 0-hour property

# scrub the request table's own raw identifiers first, then purge every table (incl. dsar_request)
spark.sql(f"""UPDATE {FQ}.dsar_request
  SET first_name={sqlstr(TOKEN)}, last_name={sqlstr(TOKEN)}, email={sqlstr(TOKEN)}, status='COMPLETE'
  WHERE status='PENDING'""")
for t in list(by_table.keys()) + ["dsar_request"]:
    purge_table(f"{FQ}.{t}", DO_PURGE)
print("purge + request scrub done")

## 4. STEP 3 — Validate: no trace of any processed subject remains
For each processed subject, scan every registered table's identifier columns (and JSON payloads) for the
subject's raw email / first / last / full name. Any hit = FAIL.

In [0]:
findings = []
for subj in erased_subjects:
    needles = {"email": subj["email"], "first": subj["first_name"],
               "last": subj["last_name"], "full": subj["first_name"]+" "+subj["last_name"]}
    for table, cols in by_table.items():
        fqt = f"{FQ}.{table}"
        for r in cols:
            col = r["column_name"]
            for label, val in needles.items():
                hits = spark.sql(
                    f"SELECT count(*) c FROM {fqt} WHERE {col} LIKE '%' || {sqlstr(val)} || '%'"
                ).collect()[0]["c"]
                if hits:
                    findings.append((subj["request_id"], table, col, label, hits))

import pandas as pd
if findings:
    print("❌ VALIDATION FAILED — residual traces found:")
    display(spark.createDataFrame(pd.DataFrame(findings,
        columns=["request_id","table","column","matched_field","rows"])))
else:
    print("✅ VALIDATION PASSED — no trace of any processed subject remains in any registered table.")

## 5. STEP 4 — Report

In [0]:
import pandas as pd
rep = pd.DataFrame(audit, columns=["request_id","table","rows_erased"])
print("Per-(request, table) rows erased:")
display(spark.createDataFrame(rep))
print("\nRequest table final state (identifiers scrubbed, status COMPLETE):")
display(spark.table(f"{FQ}.dsar_request"))

## 6. Scheduling as the monthly job
Create a Databricks **Job** with this notebook as the task and a **monthly** schedule (e.g. 1st of month).
Because the driver reads PENDING rows from `dsar_request` and the config from `pii_column_registry`, the
monthly run needs no edits — new privacy requests land in `dsar_request` (see the OneTrust intake note in the
README), new PII tables land in the registry.

**Future — OneTrust intake automation:** replace the seeded requests with a Lakeflow/REST pull from the
OneTrust *Data Subject Requests* API (OAuth bearer token) that upserts new requests into `dsar_request` with
`status='PENDING'`. Documented as a follow-up; the erasure engine is unchanged.